## Topic co-occurrence networks (STRICT + core vs non-core)

### Helper: extract topic names

In [1]:
import pandas as pd
import ast
from collections import Counter
from itertools import combinations

# --------------------------------------------------
# 1. Load STRICT works (if not loaded yet)
# --------------------------------------------------
# If you already have `strict` in memory, you can skip this line
strict = pd.read_csv("/Users/aluaaldaniyaz/Documents/Constructor studies/Data Analytics/openalex_works_strict_q1_q3_with_core_flags.csv")

print("STRICT shape:", strict.shape)
print("Columns:", list(strict.columns))

# --------------------------------------------------
# 2. Make sure topics_parsed is a proper Python object
# --------------------------------------------------

def safe_parse_json(x):
    """
    If x is a string that starts with '[' or '{', try to parse it
    into a Python object (list/dict). Otherwise return x as is.
    """
    if isinstance(x, str):
        s = x.strip()
        if s and s[0] in "[{":
            try:
                return ast.literal_eval(s)
            except Exception:
                return x
    return x

if "topics_parsed" in strict.columns:
    strict["topics_parsed"] = strict["topics_parsed"].apply(safe_parse_json)
else:
    raise ValueError("Column 'topics_parsed' not found in STRICT data.")

# --------------------------------------------------
# 3. Helper to extract topic names from topics_parsed
# --------------------------------------------------

def extract_topic_names(topics_cell):
    """
    topics_cell is usually a list of dicts like:
      [{'display_name': 'Online Learning and Analytics', 'score': 0.97, ...}, ...]
    We return a list of topic names (strings).
    """
    if isinstance(topics_cell, list):
        names = []
        for t in topics_cell:
            if isinstance(t, dict):
                name = t.get("display_name")
                if isinstance(name, str):
                    names.append(name)
        return names
    return []

# Quick check on a few rows
print("\nExample topics_parsed (first non-null row):")
print(strict["topics_parsed"].dropna().iloc[0])

print("\nExample topic_names extracted:")
print(extract_topic_names(strict["topics_parsed"].dropna().iloc[0]))

# --------------------------------------------------
# 4. Generic function to build topic node/edge tables for any subset
# --------------------------------------------------

def build_topic_network(df, suffix):
    """
    Build topic co-occurrence VOS files for a subset of STRICT works.
    
    Parameters:
      df     : pandas DataFrame (subset of STRICT)
      suffix : string used in file names, e.g. 'strict', 'core', 'noncore'
    
    Output:
      topics_map_<suffix>.txt      (tab-separated, with header)
      topics_network_<suffix>.txt  (tab-separated, NO header, as VOS prefers)
    """
    df = df.copy()
    
    # 4.1 Extract topic names per work
    df["topic_names"] = df["topics_parsed"].apply(extract_topic_names)
    
    # 4.2 Build node table: each topic and how many works it appears in
    topic_counter = Counter()
    for topics in df["topic_names"]:
        # Use set() so a topic counted only once per paper
        for name in set(topics):
            topic_counter[name] += 1
    
    nodes = pd.DataFrame(
        [{"id": name, "label": name, "weight": count}
         for name, count in topic_counter.items()]
    )
    
    # 4.3 Build edge table: topic co-occurrence (pairs per paper)
    edge_counter = Counter()
    for topics in df["topic_names"]:
        unique_topics = sorted(set(topics))
        if len(unique_topics) < 2:
            continue
        for a, b in combinations(unique_topics, 2):
            edge_counter[(a, b)] += 1
    
    edges = pd.DataFrame(
        [{"from": a, "to": b, "strength": w}
         for (a, b), w in edge_counter.items()]
    )
    
    # 4.4 Save to VOS format
    map_name = f"topics_map_{suffix}.txt"
    net_name = f"topics_network_{suffix}.txt"
    
    nodes.to_csv(map_name, sep="\t", index=False)
    # VOS often prefers no header for the edge file
    edges.to_csv(net_name, sep="\t", index=False, header=False)
    
    # 4.5 Print a small preview
    print(f"\n=== Subset: {suffix} ===")
    print("Number of works in subset:", len(df))
    print("Nodes shape:", nodes.shape)
    print("Edges shape:", edges.shape)
    
    print("\nNodes preview:")
    print(nodes.head())
    
    print("\nEdges preview:")
    print(edges.head())
    
    print("\nSaved files:")
    print(" ", map_name)
    print(" ", net_name)


# --------------------------------------------------
# 5. Build topic network for FULL STRICT
# --------------------------------------------------

build_topic_network(strict, "strict")

# --------------------------------------------------
# 6. Split into CORE and NON-CORE and build networks
# --------------------------------------------------

# We expect a boolean or 0/1 column 'is_core_work'
if "is_core_work" not in strict.columns:
    raise ValueError(
        "Column 'is_core_work' not found in STRICT data. "
        "Please add it before building core/non-core topic networks."
    )

core = strict[strict["is_core_work"] == True].copy()
noncore = strict[strict["is_core_work"] == False].copy()

print("\nCore works:", len(core), "Non-core works:", len(noncore))

build_topic_network(core, "core")
build_topic_network(noncore, "noncore")


STRICT shape: (5064, 48)
Columns: ['id', 'doi', 'title', 'abstract_inverted_index', 'publication_year', 'publication_date', 'open_access', 'type', 'language', 'cited_by_count', 'primary_location', 'best_oa_location', 'primary_topic', 'topics', 'locations', 'concepts', 'authorships', 'referenced_works', 'countries_distinct_count', 'keywords', 'counts_by_year', 'has_eu_affiliation', 'has_multiple_institutions', 'distinct_institutions', 'has_multiple_countries', 'distinct_countries', 'topics_parsed', 'concepts_parsed', 'all_issns', 'issn_list', 'issn_norm_list', 'has_any_issn', 'num_issn', 'in_scopus_flag', 'scopus_type', 'sjr_quartile', 'sjr_value', 'h_index_scimago', 'keep_q1_q3_any', 'is_strict_in_scopus', 'is_strict_q1_q3', 'source_id', 'work_id_num', 'source_id_num', 'is_core_work', 'core_pub_status', 'is_core_source', 'core_source_status']

Example topics_parsed (first non-null row):
[{'display_name': 'Artificial Intelligence in Healthcare and Education', 'score': 0.9825000166893005

In [16]:
import pandas as pd
import ast
from itertools import combinations

# for pretty previews in Jupyter
try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# 1. Load STRICT works and parse JSON-like 'authorships'
# ============================================================

# Adjust the filename if needed
strict = pd.read_csv("openalex_works_strict_q1_q3.csv")
print("STRICT shape:", strict.shape)
print("STRICT columns:", list(strict.columns))

def safe_parse_json(x):
    """
    Turn strings that look like JSON lists/dicts into real Python objects.
    We use this for 'authorships' so we can loop over authors.
    """
    if isinstance(x, str):
        s = x.strip()
        if s and s[0] in "[{":
            try:
                return ast.literal_eval(s)
            except Exception:
                return x
    return x

if "authorships" not in strict.columns:
    raise ValueError("Column 'authorships' not found in STRICT data.")

strict["authorships"] = strict["authorships"].apply(safe_parse_json)

# Make a simple alias for work_id
strict["work_id"] = strict["id"]

print("\nExample 'authorships' entry after parsing:")
print(strict["authorships"].dropna().iloc[0])


# ============================================================
# 2. Build author_long_strict: one row per (work, author)
# ============================================================

author_rows = []  # (work_id, author_id, author_name, orcid)

for _, row in strict.iterrows():
    work_id = row["work_id"]
    authorships = row.get("authorships", [])
    
    if not isinstance(authorships, list):
        continue  # safety: skip if still not list
    
    for auth in authorships:
        if not isinstance(auth, dict):
            continue
        
        author_id = auth.get("author", {}).get("id")
        author_name = auth.get("author", {}).get("display_name")
        orcid = auth.get("author", {}).get("orcid")
        
        # Only keep authors with an ID
        if author_id is None:
            continue
        
        author_rows.append((work_id, author_id, author_name, orcid))

author_long = pd.DataFrame(
    author_rows,
    columns=["work_id", "author_id", "author_name", "orcid"]
)

print("\nAuthor-long STRICT table shape:", author_long.shape)
print("Preview of author_long:")
display(author_long.head())

# Save for future reuse
author_long.to_csv("author_long_strict.csv", index=False)
print("\n✔ Saved author_long_strict.csv")


# ============================================================
# 3. Load key researchers table
# ============================================================

# Adjust filename if yours is different
key = pd.read_csv("authors_key_researchers_STRICT.csv")
print("\nKey researchers table shape:", key.shape)
print("Preview of key researchers:")
display(key.head())

# Keep only rows where is_key_researcher is True (if column exists)
if "is_key_researcher" in key.columns:
    key = key[key["is_key_researcher"] == True].copy()
    print("\nAfter filtering is_key_researcher == True:", key.shape)

key_ids = set(key["author_id"].unique())
print("Unique key researcher IDs:", len(key_ids))


# ============================================================
# 4. Build VOS NODES file for key researchers
#    Weight = citations_sum  (you can switch to works_count if you prefer)
# ============================================================

vos_nodes = key.rename(columns={
    "author_id": "id",
    "author_name": "label",
    "citations_sum": "weight"
})

vos_nodes = vos_nodes[["id", "label", "weight"]]

print("\nPreview of VOS key researcher nodes (by citations):")
display(vos_nodes.head())

vos_nodes.to_csv("VOS_key_researchers_nodes.txt", sep="\t", index=False)
print("✔ Saved VOS_key_researchers_nodes.txt")


# ============================================================
# 5. Build VOS EDGES file for key researchers
#    Co-authorships within STRICT, filtered to key IDs
# ============================================================

# Use the author_long we just built
author_core_key = author_long[author_long["author_id"].isin(key_ids)].copy()
print("\nAuthor-long restricted to key researchers:", author_core_key.shape)
print("Preview of author_core_key:")
display(author_core_key.head())

# Group by work_id to collect all key authors per STRICT paper
edges = []

for work_id, group in author_core_key.groupby("work_id"):
    authors_in_paper = list(group["author_id"].unique())
    
    # If less than 2 key authors on this paper, no co-authorship edge
    if len(authors_in_paper) < 2:
        continue
    
    # All pairs of key authors on this paper (full counting)
    for a, b in combinations(sorted(authors_in_paper), 2):
        edges.append((a, b, 1))  # +1 for this shared paper

edges_df = pd.DataFrame(edges, columns=["from", "to", "strength"])

# Aggregate across multiple shared papers
if not edges_df.empty:
    edges_df = edges_df.groupby(["from", "to"], as_index=False)["strength"].sum()

print("\nKey researcher co-authorship edges shape:", edges_df.shape)
print("Preview of edges_df:")
display(edges_df.head())

edges_df.to_csv("VOS_key_researchers_edges.txt", sep="\t", index=False, header=False)
print("✔ Saved VOS_key_researchers_edges.txt (no header, VOS-style)")


STRICT shape: (879, 41)
STRICT columns: ['id', 'doi', 'title', 'abstract_inverted_index', 'publication_year', 'publication_date', 'open_access', 'type', 'language', 'cited_by_count', 'primary_location', 'best_oa_location', 'primary_topic', 'topics', 'locations', 'concepts', 'authorships', 'referenced_works', 'countries_distinct_count', 'keywords', 'counts_by_year', 'has_eu_affiliation', 'has_multiple_institutions', 'distinct_institutions', 'has_multiple_countries', 'distinct_countries', 'topics_parsed', 'concepts_parsed', 'all_issns', 'issn_list', 'issn_norm_list', 'has_any_issn', 'num_issn', 'in_scopus_flag', 'scopus_type', 'sjr_quartile', 'sjr_value', 'h_index_scimago', 'keep_q1_q3_any', 'is_strict_in_scopus', 'is_strict_q1_q3']

Example 'authorships' entry after parsing:
[{'author_position': 'first', 'author': {'id': 'https://openalex.org/A5008809634', 'display_name': 'Enkelejda Kasneci', 'orcid': None}, 'institutions': [{'id': 'https://openalex.org/I62916508', 'display_name': 'Tech

,work_id,author_id,author_name,orcid
0,https://openalex.org/W4323655724,https://openalex.org/A5008809634,Enkelejda Kasneci,None
1,https://openalex.org/W4323655724,https://openalex.org/A5013346660,Kathrin Seßler,https://orcid.org/0000-0002-3380-4641
2,https://openalex.org/W4323655724,https://openalex.org/A5050063899,Stefan Küchemann,https://orcid.org/0000-0003-2729-1592
3,https://openalex.org/W4323655724,https://openalex.org/A5052257833,Maria Bannert,https://orcid.org/0000-0001-7045-2764
4,https://openalex.org/W4323655724,https://openalex.org/A5043273387,Daryna Dementieva,https://orcid.org/0000-0003-0929-4140



✔ Saved author_long_strict.csv

Key researchers table shape: (205, 11)
Preview of key researchers:


,author_id,author_name,orcid,works_count,citations_sum,papers_since_2020,coauthors_count,multi_institution_share,oa_share,top_concepts,is_key_researcher
0,https://openalex.org/A5029469707,Jochen Kühn,https://orcid.org/0000-0002-6985-3218,11,3920,11,55,1.0,1.0,Computer science (11); Psychology (8); Artific...,True
1,https://openalex.org/A5050063899,Stefan Küchemann,https://orcid.org/0000-0003-2729-1592,9,3901,9,50,1.0,1.0,Computer science (9); Psychology (7); Artifici...,True
2,https://openalex.org/A5052257833,Maria Bannert,https://orcid.org/0000-0001-7045-2764,6,3893,6,47,1.0,1.0,Computer science (6); Psychology (6); Artifici...,True
3,https://openalex.org/A5005995116,Frank Fischer,https://orcid.org/0000-0003-0253-659X,7,3751,7,39,1.0,1.0,Computer science (7); Psychology (5); Artifici...,True
4,https://openalex.org/A5064948864,Michael Sailer,https://orcid.org/0000-0001-6831-5429,4,3714,4,31,1.0,1.0,Computer science (4); Psychology (4); Knowledg...,True



After filtering is_key_researcher == True: (205, 11)
Unique key researcher IDs: 205

Preview of VOS key researcher nodes (by citations):


...

✔ Saved VOS_key_researchers_nodes.txt

Author-long restricted to key researchers: (137, 4)
Preview of author_core_key:


,work_id,author_id,author_name,orcid
0,https://openalex.org/W4323655724,https://openalex.org/A5008809634,Enkelejda Kasneci,None
2,https://openalex.org/W4323655724,https://openalex.org/A5050063899,Stefan Küchemann,https://orcid.org/0000-0003-2729-1592
3,https://openalex.org/W4323655724,https://openalex.org/A5052257833,Maria Bannert,https://orcid.org/0000-0001-7045-2764
5,https://openalex.org/W4323655724,https://openalex.org/A5005995116,Frank Fischer,https://orcid.org/0000-0003-0253-659X
15,https://openalex.org/W4323655724,https://openalex.org/A5023504178,Oleksandra Poquet,https://orcid.org/0000-0001-9782-816X



Key researcher co-authorship edges shape: (150, 3)
Preview of edges_df:


,from,to,strength
0,https://openalex.org/A5002876618,https://openalex.org/A5027722484,1
1,https://openalex.org/A5002876618,https://openalex.org/A5036855560,1
2,https://openalex.org/A5002876618,https://openalex.org/A5041282806,1
3,https://openalex.org/A5002876618,https://openalex.org/A5062513761,1
4,https://openalex.org/A5002876618,https://openalex.org/A5078035580,1


✔ Saved VOS_key_researchers_edges.txt (no header, VOS-style)
